# 29C — Creator benchmark label-review queue

**Runtime: CPU. Run all once.** The six external-test videos already have archived continuous landmarks. This notebook does not download video, extract pose again, train, score a model, or lock labels. It joins the 37 first-pass RGB regions and four Michelle pose-reviewed draft regions into a reproducible 41-row independent-review queue, checks structural coverage, and downloads one ZIP.

A human independent motion review is still required before Notebook 30 can evaluate frozen models. Every exported row remains `draft`, `benchmark_ready=false`, and excluded from training, calibration, threshold selection, and final evaluation.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, urllib.request, zipfile
import pandas as pd
from IPython.display import display
from google.colab import files

PINNED_COMMIT = '9f09b3dcba4376285fab0a6260821111c721d765'
RAW_BASE = f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}'
INPUTS = {
 'candidate_intervals_v1.csv': ('ml/data/youtube_creator_benchmark_v1/candidate_intervals_v1.csv','e82424985487effb9d06a8339c9be15e0008702869ae1565c3698637fcdc0381'),
 'first_pass_region_review.csv': ('ml/artifacts/youtube_creator_benchmark_v1/dense_review_evidence/first_pass_region_review.csv','d68686da30c7e0a7111286ead229f9476b8b184c615d13703b90692912a28604'),
 'supplemental_candidates_v1.csv': ('ml/data/youtube_creator_benchmark_v1/supplemental_candidates_v1.csv','ef7bd5376ab7164a824eeb2402a11f6201b1f18d163c06cdecddf1ceff563a5c'),
 'pose_contact_sheet_index.csv': ('ml/artifacts/youtube_creator_benchmark_v1/supplemental_pose_review/pose_contact_sheet_index.csv','d6d52d3d5127ec7bfa0f415cd84b4a1749fbb3b8b8b64c63fd67056a842293c1'),
 'classes.json': ('ml/config/classes.json','31bf6227d67528fc4f3fc6ecd449dfba07493dd7d1f15c57217e799837ec3207'),
}
WORK = Path('/content/calorie_creator_label_review_queue_work')
OUTPUT = Path('/content/calorie_creator_label_review_queue')
WORK.mkdir(parents=True,exist_ok=True)
if OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir()
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
paths={}
for name,(repo_path,expected_hash) in INPUTS.items():
    path=WORK/name
    if not path.exists() or sha256(path)!=expected_hash:
        temp=WORK/(name+'.part')
        temp.unlink(missing_ok=True)
        urllib.request.urlretrieve(f'{RAW_BASE}/{repo_path}',temp)
        assert sha256(temp)==expected_hash,(name,'pinned input hash changed')
        temp.replace(path)
    paths[name]=path
print('Verified',len(paths),'pinned GitHub inputs; CPU, no video download.')


## Merge candidates without pretending labels are final

First-pass decisions are navigation evidence. Rejected and unsupported candidates cannot become supported clips. The four supplemental pose regions stay provisional until an independent reviewer checks visible motion and boundaries.


In [ ]:
original=pd.read_csv(paths['candidate_intervals_v1.csv'])
first=pd.read_csv(paths['first_pass_region_review.csv'])
supplement=pd.read_csv(paths['supplemental_candidates_v1.csv'])
pose_index=pd.read_csv(paths['pose_contact_sheet_index.csv'])
classes=json.loads(paths['classes.json'].read_text())['exercises']
assert len(classes)==14 and len(original)==37 and len(first)==37 and len(supplement)==4
assert original.interval_id.is_unique and first.interval_id.is_unique and supplement.interval_id.is_unique
assert set(original.interval_id)==set(first.interval_id)
assert not set(original.interval_id)&set(supplement.interval_id)
assert set(pose_index.interval_id)==set(supplement.interval_id)
assert set(original.canonical_class).issubset(set(classes))
assert set(supplement.canonical_class).issubset(set(classes))
assert original.benchmark_ready.astype(str).str.lower().eq('false').all()
assert supplement.benchmark_ready.astype(str).str.lower().eq('false').all()
merged=original.merge(first,how='inner',on=['interval_id','video_id'],validate='one_to_one',suffixes=('','_first'))
rows=[]
for row in merged.itertuples(index=False):
    decision=row.first_pass_decision
    assert decision in {'trim_candidate','reject_candidate','relabel_unknown','variant_review','requires_split'}
    proposed=row.canonical_class
    suggested=proposed if decision=='trim_candidate' else 'unknown'
    start=row.approx_core_start_s if decision=='trim_candidate' else float('nan')
    end=row.approx_core_end_s if decision=='trim_candidate' else float('nan')
    rows.append(dict(interval_id=row.interval_id,video_id=row.video_id,creator=row.creator,
       original_proposal=proposed,first_pass_decision=decision,provisional_class=suggested,
       locator_start_s=row.start_s,locator_end_s=row.end_s,
       suggested_core_start_s=start,suggested_core_end_s=end,
       boundary_precision_s=3 if decision=='trim_candidate' else 10,
       review_basis='RGB overview: two sampled tiles per dense contact page',
       evidence_path='dense_review_evidence/dense_contact_sheets/'+row.interval_id,
       independent_review_required=True,review_status='draft',benchmark_ready=False,
       notes=row.notes_first if hasattr(row,'notes_first') else row.notes))
for row in supplement.itertuples(index=False):
    rows.append(dict(interval_id=row.interval_id,video_id=row.video_id,creator=row.creator,
       original_proposal=row.canonical_class,first_pass_decision='supplemental_pose_candidate',
       provisional_class=row.canonical_class,locator_start_s=row.start_s,locator_end_s=row.end_s,
       suggested_core_start_s=float('nan'),suggested_core_end_s=float('nan'),
       boundary_precision_s=10,review_basis='15 FPS archived pose sampled at 2 FPS + 10-second RGB context',
       evidence_path='supplemental_pose_review/pose_contact_sheets/'+row.interval_id,
       independent_review_required=True,review_status='draft',benchmark_ready=False,notes=row.notes))
queue=pd.DataFrame(rows).sort_values(['video_id','locator_start_s','interval_id']).reset_index(drop=True)
assert len(queue)==41 and queue.interval_id.is_unique and queue.video_id.nunique()==6
assert queue.review_status.eq('draft').all() and (~queue.benchmark_ready).all()
assert queue.independent_review_required.all()
assert queue.locator_end_s.gt(queue.locator_start_s).all()
valid_core=queue.suggested_core_start_s.notna()
assert queue.loc[valid_core,'suggested_core_end_s'].gt(queue.loc[valid_core,'suggested_core_start_s']).all()
queue['core_outside_original_locator']=(valid_core & ((queue.suggested_core_start_s<queue.locator_start_s) | (queue.suggested_core_end_s>queue.locator_end_s)))
print('First-pass cores outside coarse locator:',int(queue.core_outside_original_locator.sum()))
queue.to_csv(OUTPUT/'draft_review_queue.csv',index=False)
print('Draft regions:',len(queue),'| creators/videos:',queue.video_id.nunique())
display(queue.first_pass_decision.value_counts().rename('regions'))


## Coverage and second-review template

Candidate coverage is not accepted test coverage. The reviewer must inspect the linked RGB or pose evidence, set the actual class (`unknown` where needed), trim exact visible boundaries, and record uncertainty. This notebook does not auto-fill a second review or claim that independent review happened.


In [ ]:
coverage=(queue.groupby('original_proposal')
          .agg(draft_regions=('interval_id','count'),draft_creators=('video_id','nunique'),
               plausible_cores=('suggested_core_start_s',lambda s:int(s.notna().sum())))
          .reindex(classes,fill_value=0).reset_index().rename(columns={'original_proposal':'class'}))
coverage['accepted_clips']=0
coverage['benchmark_coverage_established']=False
coverage.to_csv(OUTPUT/'draft_class_coverage.csv',index=False)
review=queue[['interval_id','video_id','original_proposal','locator_start_s','locator_end_s',
              'suggested_core_start_s','suggested_core_end_s','first_pass_decision','evidence_path']].copy()
for col in ['reviewer_id','reviewed_motion_label','reviewed_start_s','reviewed_end_s',
            'segment_type','confidence','decision','notes']:
    review[col]=''
review['needs_second_review']=True
review.to_csv(OUTPUT/'independent_review_template.csv',index=False)
# Diagnostics only; no scores or label lock.
overlaps=[]
for video_id,group in queue.groupby('video_id'):
    ordered=group.sort_values('locator_start_s')
    previous=None
    for row in ordered.itertuples(index=False):
        if previous is not None and row.locator_start_s<previous.locator_end_s:
            overlaps.append(dict(video_id=video_id,earlier_id=previous.interval_id,later_id=row.interval_id,
                                 overlap_s=previous.locator_end_s-row.locator_start_s))
        previous=row
pd.DataFrame(overlaps,columns=['video_id','earlier_id','later_id','overlap_s']).to_csv(OUTPUT/'locator_overlap_audit.csv',index=False)
summary=dict(notebook='29c_colab_creator_label_review_queue.ipynb',runtime='CPU',
             pinned_commit=PINNED_COMMIT,draft_regions=len(queue),source_videos=queue.video_id.nunique(),
             supported_classes=len(classes),accepted_clips=0,labels_locked=False,
             training_allowed=False,calibration_allowed=False,threshold_selection_allowed=False,
             evaluation_allowed=False,model_predictions_reviewed=False,
             raw_video_downloaded=False,pose_extraction_performed=False,
             independent_review_completed=False,locator_overlaps=len(overlaps),
             first_pass_cores_outside_locator=int(queue.core_outside_original_locator.sum()))
(OUTPUT/'run_metadata.json').write_text(json.dumps(summary,indent=2)+'\n')
(OUTPUT/'README.md').write_text(
  '# Creator label-review queue\n\nAll 41 rows are draft. The first-pass RGB overview reviewed only two tiles per dense page; the four supplemental regions use archived pose and coarse RGB.\n\n'
  'Use independent visible-motion review to fill independent_review_template.csv. Unsupported, compound, ambiguous, rest, and transition content must be unknown. Do not use this ZIP for training, calibration, threshold selection, or Notebook 30 evaluation.\n')
for name,path in paths.items():shutil.copy2(path,OUTPUT/name)
checks=[f'{sha256(p)}  {p.relative_to(OUTPUT)}' for p in sorted(OUTPUT.rglob('*')) if p.is_file()]
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(checks)+'\n')
archive=Path(shutil.make_archive(str(OUTPUT),'zip',OUTPUT))
print('Created:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
print('SHA-256:',sha256(archive))
display(coverage)
files.download(str(archive))
